# Exploring Encodec for LGR

This notebook:
- loads HF `facebook/encodec_24khz`
- reads `.ecdc` (via your `utils.ecdc_utils.load_ecdc`)
- reads audio files and can code them as Encodec token staks or latents  
- builds a per-level lookup table **by decoding each token index** (RNeNcodec-style), used to map token stacks (or individual tokens) to latents.

- Big picture:
  1) Creates latent "pool" used for decoding
  2) Codes target audio as latents
  3) For each (window size) of target latents, finds closest codes in "pool" and uses the mapped latents for decoding

(Analysis of cos vs L2 distance follows at the end (after red line)  

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import EncodecModel

from IPython.display import Audio, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "facebook/encodec_24khz"

model = EncodecModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", DEVICE)
print("Model sampling rate:", getattr(model.config, "sampling_rate", None))
print("Codebook size (K):", getattr(model.config, "codebook_size", None))
print("Target bandwidths:", getattr(model.config, "target_bandwidths", None))

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Loaded: facebook/encodec_24khz
Device: cpu
Model sampling rate: 24000
Codebook size (K): 1024
Target bandwidths: [1.5, 3.0, 6.0, 12.0, 24.0]


In [2]:
from utils.ecdc_utils import load_wav_mono
from utils.ecdc_utils import load_ecdc  # read a stored ecdc file
from utils.ecdc_utils import  encode_audio_to_tokens #convert an audio to an ecdc representation

from utils.ecdc_utils import bandwidth_to_n_q, n_q_to_bandwidth

# Creates the lookup table for an Encodec model to use for token-> latent mapping
from utils.ecdc_utils import  build_LOOKUP_via_layer_decode

# Creates a "pool" of tokens that can be drawn on for style transfer purposes
from utils.ecdc_utils import  tokens_to_summary_latents

# returns tensor of latents for just one level of a token stack sequence
from utils.ecdc_utils import  token_level_to_latents

# take a target audio segment to a tensor of encodec latents
from utils.ecdc_utils import  audio_to_latents

from utils.ecdc_utils import  latents128_to_audio, tokens_TN_to_audio_1T
#=======================
# imports specifically for the Tokui-like style transfer

from utils.tokui_utils import normalize_latents, target2poolindex 
from utils.tokui_utils import gather_tokens_by_index 

<div style="width: 100%; height: 2px; background-color: red;"></div>
Tensor ordering is a mess between HF (Hugging Face, Encodec, and canonical). <br>
<span style="color: red;">
      - "TN":  [T_frames, n_q]      (recommended canonical)  <br>  
      - "BQT": [B, n_q, T_frames]   (HuggingFace-friendly)  <br>  
      - "QBT": [n_q, B, T_frames]   (for model.quantizer.decode)  <br>  
</span>

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Utility Functions for your coding pleasure</b><br>

<span style="color: blue;">
Translate between Encodec "bandwidth" representations
bandwidths (kbs): [1.5, 3.0, 6.0, 12.0, 24.0] <->  number of codebooks (n_q): [2, 4, 8, 16, 32]  
</span> <br>
bandwidth_to_n_q(bw_kbps)  
n_q_to_bandwidth(n_q)   
<br><br>

<span style="color: blue;">
Builds the datastructure for mapping tokens to latents   
</span> <br>
build_LOOKUP_via_layer_decode(model, n_q, K_codbook_size, device=DEVICE)
<br><br>

<span style="color: blue;">
Uses the loopkup table datastructure to map tokens to latents   
</span> <br>
tokens_to_summary_latents(tokens_TN, LOOKUP_QKD )    <br>
(First arg in (T, N) form, Second arg is datastructure mapping tokens to latents)  
<br><br>



<span style="color: blue;">
Returns latents for a single cobook level of codes 
</span> <br>
token_level_to_latents(tokens_TN: torch.Tensor, level_q: int, LOOKUP_QKD: torch.Tensor)    
<br><br>

<span style="color: blue;">
Encodes an audio signal to tokens
</span> <br>
encode_audio_to_tokens(audio, model, DEVICE, n_q_to_bandwidth(8))  
<br><br>

<span style="color: blue;">
Returns latents for an audio signal
</span> <br>
audio_to_latents(audio, model, device: str, bandwidth)  
<br><br>


<span style="color: blue;">
(T,N) tensor of latents to audio  
</span> <br>
latents128_to_audio(model, z_T128, device)  
<br><br>

<span style="color: blue;">
(T,N) tensor of tokens to audio  
</span> <br>
tokens_TN_to_audio_1T(model, tokens_TN: torch.Tensor, device, audio_scales=None, last_frame_pad_length: int = 0)  
<br><br>



In [3]:

if 0: # Load the "pool" from an.ecdc file (tokens_TN is (T, n_q)) ----
    dspath = "/slowdisk/data/syn7_v4_sm/tokens/validation/DSPeepers--max_range-00.55--c-02--x-98.ecdc"
    pool_tokens_TN, scales, raw = load_ecdc(dspath)
else : # OR from audio 
    #dspath = "wav24k/amen_mono_24k.wav" # "wav24k/diverse24.wav"
    dspath = "wav24k/pans24.wav"
    pool_audio=load_wav_mono(dspath)
    pool_tokens_TN, raw = encode_audio_to_tokens(pool_audio, model, DEVICE, n_q_to_bandwidth(8))
    scales = getattr(raw, "audio_scales", None)


encode_audio_to_tokens with n_q=8


In [4]:
#show some infor about the pool
n_q= pool_tokens_TN.shape[1]
print("pool_tokens_TN:", pool_tokens_TN.shape, pool_tokens_TN.dtype)   # (T, n_q)
print("n_q:", n_q)
print("scales:", scales)
print("raw keys:", list(raw.keys()))
print("audio_length:", raw.get("audio_length"))

pool_tokens_TN: torch.Size([9240, 8]) torch.int64
n_q: 8
scales: [None]
raw keys: ['audio_codes', 'audio_scales', 'last_frame_pad_length']
audio_length: None


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Create the lookup table for using in function taking codes -> latents </b>

In [5]:
# Create lookup table (with same number of codebooks as poos) for fast token->latent mapping
K = int(getattr(model.config, "codebook_size", 1024))
n_q_data = int(pool_tokens_TN.shape[1])
LOOKUP_QKD = build_LOOKUP_via_layer_decode(model, n_q=n_q_data, K=K, device=DEVICE)
print("LOOKUP_QKD:", LOOKUP_QKD.shape, LOOKUP_QKD.dtype, LOOKUP_QKD.device)

LOOKUP_QKD: torch.Size([8, 1024, 128]) torch.float32 cpu


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Create POOL  latents</b>

In [6]:
# now use the lookup table to first create the "pool" of latents
pool_latents = tokens_to_summary_latents(pool_tokens_TN, LOOKUP_QKD)
print("pool_latents:", pool_latents.shape, pool_latents.dtype, pool_latents.device)


Taking tokens_to_summary_latents with n_q=8
pool_latents: torch.Size([9240, 128]) torch.float32 cpu


In [7]:
# listen to the pool (coded and decodec)
pool_latents_audio = latents128_to_audio(model, pool_latents, DEVICE).cpu()
display(Audio(pool_latents_audio, rate=24000))

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Create TARGET latents</b>

In [8]:
target_audio=load_wav_mono("wav24k/amen_mono_24k.wav")
print(f'DEVICE is {DEVICE}')
target_latents, _, _ = audio_to_latents(target_audio, model, DEVICE, n_q_to_bandwidth(n_q))
print("target_latents:", target_latents.shape, target_latents.dtype, target_latents.device)

# lets just make sure when we convert the latents back to audio it sound right.
test_target_audio= latents128_to_audio(model, target_latents, DEVICE).cpu()
display(Audio(test_target_audio, rate=24000))

DEVICE is cpu
encode_audio_to_tokens with n_q=8
target_latents: torch.Size([524, 128]) torch.float32 cpu


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>DO MATCHING ! </b>

The pool vectors should be normalized already, the target vectors get normalized in the matching method (we are anticipating that they come in real time). 
<b>The window is the size is used thus:<b>  
- Slide a n-frame window across the pool.  
- For each window starting on index p, 
score(p) = sum_i cosine(target[t+i], pool[p+i])  
- The best scoring window (of p vectors) is returned as the match for the p target vectors.  f

(larger window size yields greater sense of "pool" audio)

In [9]:
#This is the key: find pool latents closest (cos angle) to target latents
# Note: the pool vectors must be ALREADY normalized
pool_latents_normed= normalize_latents(pool_latents)
pool_indices=target2poolindex(target_latents, pool_latents_normed, window=7)

In [10]:
# print the pool indices used
if 0 :   #if 1: to see the codebook indeces for pool
    pool_indices

In [11]:
st_tokens_TN = gather_tokens_by_index(pool_tokens_TN, pool_indices)  # [T_target, N]

In [12]:
st_audio = tokens_TN_to_audio_1T(model, st_tokens_TN, DEVICE).cpu()

In [13]:
display(Audio(st_audio, rate=24000))

<div style="width: 100%; height: 40px; background-color: red;"></div>
<b>COMPARE cos similarity to distancy measure for choosing vectors </b>
 

In [14]:
from utils.teaching_utils import  target2poolindex_cos, target2poolindex_l2, plot_match_scores_normalized

ModuleNotFoundError: No module named 'utils.teaching_utils'

In [ ]:
window_length=4

<div style="width: 100%; height: 5px; background-color: pink;"></div>
<b> cos </b>

In [ ]:
pool_indices_cos, dcos =target2poolindex_cos(target_latents, pool_latents_normed, window=window_length)
st_tokens_cos = gather_tokens_by_index(pool_tokens_TN, pool_indices_cos)  # [T_target, N]
st_audio_cos = tokens_TN_to_audio_1T(model, st_tokens_cos, DEVICE).cpu()
display(Audio(st_audio_cos, rate=24000))

In [ ]:
# print the indices chosen
#pool_indices_cos

<div style="width: 100%; height: 5px; background-color: pink;"></div>
<b> L2 </b>

In [ ]:
# pool tokens NOT normalized for L2 computations
pool_indices_l2, dl2 =target2poolindex_l2(target_latents, pool_latents, window=window_length)
st_tokens_l2 = gather_tokens_by_index(pool_tokens_TN, pool_indices_l2)  # [T_target, N]
st_audio_l2 = tokens_TN_to_audio_1T(model, st_tokens_l2, DEVICE).cpu()
display(Audio(st_audio_l2, rate=24000))

In [ ]:
# print the indices chosen
#pool_indices_l2

<div style="width: 100%; height: 5px; background-color: pink;"></div>
<b> Now compare the distances between target and chosen vectors in cos and L2 </b>

In [ ]:
plot_match_scores_normalized(dcos.cpu(), dl2.cpu())

so why is L2 working (and looking) about the same as cos similarity?

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_norm_distributions(target_raw, pool_raw, bins=60, title="Latent norm distributions"):
    target_raw = target_raw.detach().float().cpu()
    pool_raw   = pool_raw.detach().float().cpu()

    tn = torch.linalg.norm(target_raw, dim=1).numpy()
    pn = torch.linalg.norm(pool_raw,   dim=1).numpy()

    def stats(x):
        return dict(mean=float(np.mean(x)), std=float(np.std(x)),
                    cv=float(np.std(x)/(np.mean(x)+1e-12)),
                    min=float(np.min(x)), max=float(np.max(x)))

    print("Target norms:", stats(tn))
    print("Pool norms:  ", stats(pn))

    plt.figure()
    plt.hist(tn, bins=bins, alpha=0.6, label="target")
    plt.hist(pn, bins=bins, alpha=0.6, label="pool")
    plt.xlabel("L2 norm ||z||")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.show()


In [ ]:
plot_norm_distributions(target_latents, pool_latents)

In [ ]:
@torch.no_grad()
def plot_cos_vs_l2_for_pairs(target_raw, pool_raw, idx_T, n_points=2000, title="Cosine vs L2 (matched pairs)"):
    target_raw = target_raw.detach().float().cpu()
    pool_raw   = pool_raw.detach().float().cpu()
    idx_T      = idx_T.detach().long().cpu()

    T = target_raw.shape[0]
    n = min(n_points, T)
    sel = torch.randperm(T)[:n]

    a = target_raw[sel]               # [n,D]
    b = pool_raw[idx_T[sel]]          # [n,D]

    # cosine (use normalized copies)
    a_n = torch.nn.functional.normalize(a, dim=1)
    b_n = torch.nn.functional.normalize(b, dim=1)
    cos = (a_n * b_n).sum(dim=1).numpy()

    # raw L2
    l2 = torch.linalg.norm(a - b, dim=1).numpy()

    # correlation
    corr = float(np.corrcoef(cos, l2)[0,1])
    print(f"Pearson corr(cos, L2) = {corr:.4f}  (expect strong negative if they track)")

    plt.figure()
    plt.scatter(cos, l2, s=8)
    plt.xlabel("Cosine similarity (higher is better)")
    plt.ylabel("Raw L2 distance (lower is better)")
    plt.title(title)
    plt.show()


In [ ]:
plot_cos_vs_l2_for_pairs(target_latents, pool_latents, pool_indices_cos)


In [ ]:
@torch.no_grad()
def plot_l2_decomposition(target_raw, pool_raw, idx_T, n_points=2000, title="L2² decomposition"):
    target_raw = target_raw.detach().float().cpu()
    pool_raw   = pool_raw.detach().float().cpu()
    idx_T      = idx_T.detach().long().cpu()

    T = target_raw.shape[0]
    n = min(n_points, T)
    sel = torch.randperm(T)[:n]

    x = target_raw[sel]
    y = pool_raw[idx_T[sel]]

    nx = torch.linalg.norm(x, dim=1)
    ny = torch.linalg.norm(y, dim=1)

    x_n = torch.nn.functional.normalize(x, dim=1)
    y_n = torch.nn.functional.normalize(y, dim=1)
    cos = (x_n * y_n).sum(dim=1).clamp(-1, 1)

    l2_sq = torch.sum((x - y) ** 2, dim=1)

    norm_term  = (nx - ny) ** 2
    angle_term = 2 * nx * ny * (1 - cos)

    # sanity: l2_sq ≈ norm_term + angle_term
    err = torch.max(torch.abs(l2_sq - (norm_term + angle_term))).item()
    print("max decomposition error:", err)

    # plot distributions
    plt.figure()
    plt.hist(l2_sq.numpy(), bins=60, alpha=0.6, label="L2²")
    plt.hist(norm_term.numpy(), bins=60, alpha=0.6, label="(||x||-||y||)²")
    plt.hist(angle_term.numpy(), bins=60, alpha=0.6, label="2||x||||y||(1-cos)")
    plt.xlabel("Value")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.show()

    # print typical proportions
    frac = (norm_term / (l2_sq + 1e-12)).numpy()
    print(f"Median fraction of L2² from norm term: {float(np.median(frac)):.3f}")
    print("90th percentile norm fraction:", float(np.percentile(frac, 90)))
    print("99th percentile norm fraction:", float(np.percentile(frac, 99)))


In [ ]:
plot_l2_decomposition(target_latents, pool_latents, pool_indices_cos)

In [ ]:
@torch.no_grad()
def norm_vs_cos_scatter(latents, n_samples=5000):
    import numpy as np
    import matplotlib.pyplot as plt
    
    lat = latents.detach().float().cpu()
    N = lat.shape[0]
    
    idx1 = torch.randint(0, N, (n_samples,))
    idx2 = torch.randint(0, N, (n_samples,))
    
    x = lat[idx1]
    y = lat[idx2]
    
    nx = torch.linalg.norm(x, dim=1)
    ny = torch.linalg.norm(y, dim=1)
    
    x_n = torch.nn.functional.normalize(x, dim=1)
    y_n = torch.nn.functional.normalize(y, dim=1)
    cos = (x_n * y_n).sum(dim=1)
    
    plt.figure()
    plt.scatter((nx - ny).abs().numpy(), cos.numpy(), s=5)
    plt.xlabel("|norm(x) - norm(y)|")
    plt.ylabel("cos(x,y)")
    plt.title("Norm difference vs cosine similarity")
    plt.show()

In [ ]:
norm_vs_cos_scatter(pool_latents)

In [ ]:
# ---- 2b) Individual level contribution (T,128) ----
# @torch.no_grad()
# def token_level_to_latents(tokens_TN: torch.Tensor, level_q: int, LOOKUP_QKD: torch.Tensor) -> torch.Tensor:
#     tokens_TN = tokens_TN.to(LOOKUP_QKD.device, dtype=torch.long)
#     return LOOKUP_QKD[level_q][tokens_TN[:, level_q]]  # (T,128)

# # Example: level 0
# z0 = token_level_to_latents(tokens_TN, 0, LOOKUP_QKD)
# print("z0:", z0.shape)


## Notes

- **HF expects `quantizer.decode(codes)` input shape `(n_q, B, T)`**, not `(B, n_q, T)`.
- The lookup table is built **by calling `layers[q].decode(idx_all)`**, matching the RNeNcodec approach for creating index->latents.
- Reference:
```text
Tokui, N., & Baker, T. (2025). Latent Granular Resynthesis using Neural Audio Codecs. arXiv preprint arXiv:2507.19202.
```